In [1]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import numpy as np
from load_results import load
df = load()

/mnt/nlp-data/home/users/loriss21cs/eemc/load_results.py:60: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["k"] = df["k"].fillna(-1)


In [2]:
A_df = (
    df
    .groupby(["score_name", "base_model", "Experiment", "Rep_A", "Rep_B", "test_type", "k"])["score"]
    .mean()
    .unstack("Experiment", "Rep_A", "Rep_B")
)

long_df = (
    A_df.reset_index()
     .melt(
         id_vars=["score_name", "base_model", "Rep_A", "Rep_B", "test_type", "k"],
         var_name="dataset",
         value_name="score"
     )
     .rename(columns={"score_name": "metric"})
)
long_df["Rep_pair"] = long_df.apply(
    lambda r: ":".join(sorted([str(r["Rep_A"]), str(r["Rep_B"])])),
    axis=1
)


# long_df = long_df[~((long_df["metric"].isin(["cycle_knn", "cknna", "f1_at_k"])) & (long_df["k"] == 10))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["ndcg", "ndcg_k", "spearman_rho"])) & (long_df["k"].isin([-1,10])))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["rbo"])) & (long_df["k"].isin([10])))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["sign_agreement", "unbiased_cka"])) & (long_df["k"].isin([1,-1,10])))] # TODO
long_df.head()

,metric,base_model,Rep_A,Rep_B,test_type,k,dataset,score,Rep_pair
0,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r^{id_{X}}_{\nabla f}$,rep,10,code,1.000000,$r^{L2}_{\nabla f}$:$r^{id_{X}}_{\nabla f}$
1,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{\nabla f}^{aa}$,rep,10,code,0.866309,$r^{L2}_{\nabla f}$:$r_{\nabla f}^{aa}$
2,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{s_{{test}}}$,rep,10,code,0.406855,$r^{L2}_{\nabla f}$:$r_{s_{{test}}}$
3,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{attn}}$,rep,10,code,0.080853,$r^{L2}_{\nabla f}$:$r_{{attn}}$
4,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{bm25}}$,rep,10,code,0.094380,$r^{L2}_{\nabla f}$:$r_{{bm25}}$


## fit models

$y_{i,m,d}=\beta_0+\beta^{(m)}_{m}+\beta^{(d)}_{d}+\beta^{(m \times d)}_{m,d}+u_i+\epsilon_{i,m,d}$

In [3]:
def fit_model(definition, long_df, score_name, k=-1):
    df = long_df[(long_df["metric"] == score_name) & (long_df["k"] == k)].copy()
    cols = ["base_model", "dataset", "score", "Rep_pair"]
    if df[cols].isnull().values.any():#, f"missing entries for {score_name} at k={k}":
        return None
    if (df[cols].nunique() == 1).any():
        return None
    return smf.mixedlm(
                definition,
                data=df[cols],
                groups=df["Rep_pair"]
            ).fit()

configs = long_df.groupby(["metric", "k"]).groups.keys()
models = {
    (score_name, k) : fit_model("score ~ C(base_model) * C(dataset)", long_df, score_name, k) for score_name, k in configs
} 

models_no_interaction = {
    (score_name, k) : fit_model("score ~ C(base_model) + C(dataset)", long_df, score_name, k) for score_name, k in configs
} 

models_no_base_model = {
    (score_name, k) : fit_model("score ~ C(dataset)", long_df, score_name, k) for score_name, k in configs
} 

models_no_dataset = {
    (score_name, k) : fit_model("score ~ C(base_model)", long_df, score_name, k) for score_name, k in configs
} 


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warn

## hypothesis testing

In [4]:
# interaction_results = []
# for (score_name, k), model in models.items():
#     if model is None:
#         continue
#     params = model.params.index
#     interaction_terms = [p for p in params if ":" in p]
#     R = np.zeros((len(interaction_terms), len(params)))
#     for i, term in enumerate(interaction_terms):
#         R[i, params.get_loc(term)] = 1
#     wald_test = model.wald_test(R, scalar=True)
#     interaction_results.append((score_name, k, wald_test.statistic, wald_test.pvalue, wald_test.df_denom))
# df_interaction = pd.DataFrame(interaction_results, columns=["metric", "k", "$\chi^2$", "pval", "denom"]).set_index(["metric", "k"])
# df_interaction

$H_0: \beta^{(m \times d)}_{m,d} = 0 \quad \forall m,d,$



$H_0^{(M)}: \beta^{(m)}_{m} = 0 \quad \forall m$

$H_0^{(D)}: \beta^{(d)}_{d} = 0 \quad \forall d$

In [20]:
def compute_r2_mixedlm(model):
    # fixed effects design matrix
    fe = model.model.exog
    fe_params = model.fe_params.values

    # fixed-effect linear predictor η
    eta = fe @ fe_params
    sigma_f2 = np.var(eta, ddof=1)

    # random intercept variance (Rep_pair)
    sigma_pair2 = model.cov_re.values.item()

    # residual variance
    sigma_e2 = model.scale

    denom = sigma_f2 + sigma_pair2 + sigma_e2

    R2_m = sigma_f2 / denom
    R2_c = (sigma_f2 + sigma_pair2) / denom

    return sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c


result_dfs_wald = []
result_dfs_r2 = []
for (score_name, k), model in models.items():
    if model is None:
        continue
    interaction_test = model.wald_test_terms(scalar=True)
    df_model_dataset = interaction_test.summary_frame().stack().rename_axis(["term", "stat"]).to_frame().T.reset_index(drop=True)
    df_model_dataset["metric"] = score_name
    df_model_dataset["k"] = k
    df_model_dataset = df_model_dataset.set_index(["metric", "k"])
    result_dfs_wald.append(df_model_dataset)
    
    r2_dfs = []
    for setting, model_ in [
        ("full", model),
        ("no_interaction", models_no_interaction[(score_name, k)]),
        ("no_base_model", models_no_base_model[(score_name, k)]),
        ("no_dataset", models_no_dataset[(score_name, k)])
        ]:
        sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c = compute_r2_mixedlm(model_)
        df_r2 = pd.DataFrame({

            "metric": [score_name],
            "k": [k],
            # "sigma_f2": [sigma_f2],
            # "sigma_pair2": [sigma_pair2],
            # "sigma_e2": [sigma_e2],
            "R2_m": [R2_m],
            "R2_c": [R2_c]
        })
        df_r2.columns = pd.MultiIndex.from_product(
            [[setting], df_r2.columns]
        )

        r2_dfs.append(df_r2)
    result_dfs_r2.append(pd.concat(r2_dfs, axis=1))
        
    

    
 
df_wald = pd.concat(result_dfs_wald)
df_r2_model_dataset = pd.concat(result_dfs_r2)

display(df_wald)
# df_r2_model_dataset

term                  Intercept                              C(base_model)  \
stat                       chi2         P>chi2 df constraint          chi2   
metric         k                                                             
cka            10    112.685969   2.528051e-26           1.0  7.777634e-01   
cknna          10     60.798144   6.323753e-15           1.0  1.774803e+00   
cycle_knn      10    361.262550   1.495094e-80           1.0  1.327204e+01   
f1_at_k        10     34.418462   4.444870e-09           1.0  3.569897e-02   
ndcg           10  12193.301048   0.000000e+00           1.0  1.123725e+00   
ndcg_k         10    779.319387  1.692280e-171           1.0  1.983921e-01   
rbo            10     33.315541   7.835395e-09           1.0  1.074219e+00   
sign_agreement 10    553.953660  1.737457e-122           1.0  4.408096e-13   
spearman_rho   10      4.098148   4.293022e-02           1.0  2.758333e-01   
unbiased_cka   10     77.018143   1.693946e-18           1.0  9.227752e-01   

term                                       C(dataset)                \
stat                 P>chi2 df constraint        chi2        P>chi2   
metric         k                                                      
cka            10  0.377826           1.0   47.246780  1.354690e-09   
cknna          10  0.182789           1.0   81.690382  7.634921e-17   
cycle_knn      10  0.000269           1.0  152.270920  6.638101e-32   
f1_at_k        10  0.850139           1.0  122.132210  1.871466e-25   
ndcg           10  0.289118           1.0  127.431112  1.379413e-26   
ndcg_k         10  0.656022           1.0    7.467628  1.131453e-01   
rbo            10  0.299994           1.0  106.079850  4.986180e-22   
sign_agreement 10  0.999999           1.0    1.042046  9.033548e-01   
spearman_rho   10  0.599445           1.0    8.971411  6.181805e-02   
unbiased_cka   10  0.336747           1.0   56.544573  1.541553e-11   

term                            C(base_model):C(dataset)                \
stat              df constraint                     chi2        P>chi2   
metric         k                                                         
cka            10           4.0                 3.059709  5.478835e-01   
cknna          10           4.0                 1.724079  7.863391e-01   
cycle_knn      10           4.0                26.058444  3.079742e-05   
f1_at_k        10           4.0                 1.679274  7.944791e-01   
ndcg           10           4.0                39.780225  4.805900e-08   
ndcg_k         10           4.0                 2.977878  5.615342e-01   
rbo            10           4.0                 1.102405  8.938905e-01   
sign_agreement 10           4.0                 0.030403  9.998856e-01   
spearman_rho   10           4.0                 1.362725  8.506470e-01   
unbiased_cka   10           4.0                 3.480725  4.808152e-01   

term                             
stat              df constraint  
metric         k                 
cka            10           4.0  
cknna          10           4.0  
cycle_knn      10           4.0  
f1_at_k        10           4.0  
ndcg           10           4.0  
ndcg_k         10           4.0  
rbo            10           4.0  
sign_agreement 10           4.0  
spearman_rho   10           4.0  
unbiased_cka   10           4.0

In [21]:
df = df_r2_model_dataset.copy()

df = df.set_index(("full", "metric"))

delta_r2 = pd.DataFrame(index=df.index)

delta_r2["interaction_model_dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_interaction", "R2_m")]
)
delta_r2["interaction_model_dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_interaction", "R2_c")]
)

delta_r2["base_model_R2_m"] = (
    df[("full", "R2_m")] - df[("no_base_model", "R2_m")]
)
delta_r2["base_model_R2_c"] = (
    df[("full", "R2_c")] - df[("no_base_model", "R2_c")]
)

delta_r2["dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_dataset", "R2_m")]
)
delta_r2["dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_dataset", "R2_c")]
)
delta_r2

,interaction_model_dataset_R2_m,interaction_model_dataset_R2_c,base_model_R2_m,base_model_R2_c,dataset_R2_m,dataset_R2_c
"(full, metric)",,,,,,
cka,8.794517e-04,-0.000079,3.939100e-03,0.003069,0.024056,0.024647
cknna,2.518210e-04,-0.000333,1.630975e-03,0.001054,0.025876,0.027532
cycle_knn,7.555225e-03,0.007464,2.823297e-02,0.030257,0.074236,0.080537
f1_at_k,2.691208e-04,-0.000143,6.299099e-04,0.000148,0.036671,0.039879
ndcg,1.839800e-03,0.001867,3.406653e-03,0.003564,0.019814,0.021678
ndcg_k,7.436893e-05,-0.000022,1.896003e-04,0.000080,0.000296,0.000119
rbo,8.418190e-05,-0.000195,9.368891e-04,0.000681,0.015522,0.016697
sign_agreement,7.515183e-07,-0.000104,9.394297e-07,-0.000130,0.000066,-0.000136
spearman_rho,1.901498e-04,-0.000213,4.007606e-04,-0.000085,0.002111,0.001496


### 